In [ ]:
%gui qt
import json
import numpy as np
import fastplotlib as fpl
import wgpu

import pygfx # for typing?

In [ ]:
file_path = "diatribes_shaders.json"
# file_path = "jakel101_shaders.json"
with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print(len(data["shaders"]))

In [ ]:
# as placeholders, untill we have a single "custom" shape?

# https://docs.pygfx.org/stable/_autosummary/materials/pygfx.materials.PointsMarkerMaterial.html
def map_shapes(tags) -> str:
    if "tunnel" in tags:
        return "💎"
    elif "clouds" in tags:
        return "❤️"
    else:
        return "●"

# is there a proper reference?
def map_colors(tags) -> str:
    if "raymarching" in tags:
        return "blue"
    elif "raymarch" in tags:
        return "purple"
    elif "fractal" in tags:
        return "green"
    else:
        return "red"

In [ ]:
# get char number form title
import re
# name [123] or name [ 1 23 ] etc
chars_pattern = re.compile(r"\[(\s*\d+\s*)+\]")
def get_chars(title:str) -> int:
    match = chars_pattern.search(title)
    if match:
        num_str = match.group(1)
        num_str = num_str.replace(" ", "")
        return int(num_str)
    return -1
    

In [ ]:
# extract data into python objects
username = data["userName"]
str_len = [len(shader_data["renderpass"][0]["code"]) for shader_data in data["shaders"]]
chars = [get_chars(shader_data["info"]["name"]) for shader_data in data["shaders"]]
dates = [int(shader_data["info"]["date"]) for shader_data in data["shaders"]]
likes = [shader_data["info"]["likes"] for shader_data in data["shaders"]]
views = [shader_data["info"]["viewed"] for shader_data in data["shaders"]]
ids = [shader_data["info"]["id"] for shader_data in data["shaders"]]
shapes = [map_shapes(shader_data["info"]["tags"]) for shader_data in data["shaders"]]
colors = [map_colors(shader_data["info"]["tags"]) for shader_data in data["shaders"]]
info = np.array([chars, likes, views]).T
info

In [ ]:
### from: https://github.com/pygfx/rendercanvas/pull/153
import rendercanvas
from wgpu_shadertoy import Shadertoy
from rendercanvas.offscreen import OffscreenRenderCanvas
import pygfx as gfx
from pygfx.renderers.wgpu.engine.update import ensure_wgpu_object


class WgpuContext(rendercanvas.contexts.WgpuContext):
    def __new__(cls, present_info: dict, texture: gfx.Texture):
        # avoid the special behaviour?
        return object.__new__(cls)

    def __init__(self, present_info: dict, texture: gfx.Texture):
        # TODO: can you initialize this with a pygfx texture too?
        self.texture = texture
        # TODO: ensure this is a render attachment usage, and then refreush it's wgpu object?
        self._config = None # should already exist tho
        super().__init__(present_info)

    def _get_preferred_format(self, adapter=None) -> str:
        wgpu_format = gfx.renderers.wgpu.to_texture_format(self.texture.format)
        # so any kind of render pipeline setup by canvas like use is compatible with the texture we will provide
        return wgpu_format

    def configure(
        self,
        *,
        device: wgpu.GPUDevice,
        format: str,
        usage: str | int = "RENDER_ATTACHMENT",
        view_formats = (),
        alpha_mode: str = "opaque",) -> None:
        # TODO: make
        inp_config = wgpu.CanvasConfiguration(device=device, format=format, usage=usage, view_formats=view_formats, alpha_mode=alpha_mode) # just to make it type correctly
        assert inp_config.device == gfx.renderers.wgpu.get_shared().device, "you need to use the same device!"
        assert inp_config.format == self._get_preferred_format(), "the target format needs to be that of the existing texture, use the preferred format only!"
        # inp_config.usage |= wgpu.TextureUsage.RENDER_ATTACHMENT #always 0x10 anyway?
        # TODO: can we just ignore the rest?
        self._config = inp_config

    def _unconfigure(self) -> None:
        # don't think I need or want this?
        pass

    def _get_current_texture(self) -> wgpu.GPUTexture:
        if self.texture._wgpu_object is None:
            ensure_wgpu_object(self.texture)
        return self.texture._wgpu_object

    def _rc_present(self) -> None:
        # make ready for sync or something, so the actual pygfx renderer can use it!
        self.texture._gfx_mark_for_sync()
        return {"method": "screen"}

    def _rc_bitmap_present(self) -> None:
        self._rc_present()

    @property
    def physical_size(self) -> tuple[int, int]:
        return self.texture.size[0], self.texture.size[1]

texture1 = gfx.Texture(
    size=(512, 512, 1),
    dim=2,
    format="rgba8unorm",
    usage=wgpu.TextureUsage.RENDER_ATTACHMENT | wgpu.TextureUsage.TEXTURE_BINDING,
)

# the simplest wrapper, which likely produces a bunch of parts we don't need.
# TODO: actually translate world space events from pygfx into rendercanvas clicks and resizes etc (should be possible?)
class OffscreenPygfxRenderCanvas(OffscreenRenderCanvas):
    def __init__(self, texture: gfx.Texture, *args, **kwargs):
        self._canvas_context = WgpuContext(present_info={"method": "screen"}, texture=texture)
        kwargs["size"] = texture.size[0], texture.size[1]
        super().__init__(*args, **kwargs)

canvas1 = OffscreenPygfxRenderCanvas(texture=texture1) # the "offscreen" canvas that our external app will render to

In [ ]:
# for the preview
def load_shadertoy(shader_data: dict, resolution=(512, 512), canvas=canvas1, device=None) -> Shadertoy:
    shader_data["info"]["username"] = username
    shader_data = scrape_to_api(shader_data)
    kwargs = {"resolution": resolution, "canvas": canvas, "device": device}
    st = Shadertoy.from_json(shader_data, **kwargs)
    return st

# from shadertoys-dataset/download.py https://github.com/Vipitis/shadertoys-dataset/blob/main/download.py
def scrape_to_api(json_data: dict) -> dict:
    """
    transform the dict to be exactly like the API return would provide it
    """
    privacy_keys = {0: "Private", 1: "Public", 2: "Unlisted", 3: "Public API", 4: "Anonymous"}
    
    shader_data = {
        "Shader": {
            "info": json_data["info"],
            "ver": json_data["ver"],
            "renderpass": json_data["renderpass"],
        }
    }
    # del shader_data["Shader"]["info"]["usePreview"] # indicates if a shader is "heavy" and should not be rendered in preview. Maybe useful for filtering?
    for rp in shader_data["Shader"]["renderpass"]:
        for inp in rp["inputs"]:
            inp["src"] = inp.pop("filepath")
            inp["ctype"] = inp.pop("type")

    # TODO: that seems be be incorrect, download gives these. scrape and API gives numbers -.-
    shader_data["Shader"]["info"]["published"] = privacy_keys.get(shader_data["Shader"]["info"]["published"], "Unknown")

    return shader_data


In [ ]:
%gui qt
# https://www.fastplotlib.org/ver/dev/_gallery/scatter/scatter.html#sphx-glr-gallery-scatter-scatter-py
figure = fpl.Figure(size=(1100, 1000), shape=(2, 1))
fpl_device = figure.renderer._device # wgpu device used by fastplotlib
# TODO: title?
scatter_plot = figure[0, 0].add_scatter(
    data=info[:, :2],
    sizes=(info[:, 2] / 100) + 4, #sorta a min scale
    colors=colors,
    markers=shapes,
    alpha=0.4,
    uniform_edge_color=False,
)
figure[0, 0].auto_scale(maintain_aspect=False)

# have the shadertoy running in the lower half
PREVIEW_RES = (512, 512) # TODO: get this from where? # TODO: flipped axis


# TODO: make a custom graphics class so we don't initialize all of this first?
random_frame = np.random.randint(0, 255, (*PREVIEW_RES, 4)).astype(np.float32)
preview = figure[1, 0].add_image(random_frame)
figure[1, 0].axes.visible = False
# needs to be initialized for the update function to work
shader = load_shadertoy(data["shaders"][0], PREVIEW_RES, device=fpl_device, canvas=canvas1)

def update_preview(subplot):
    shader._draw_frame()

figure[1, 0].add_animations(update_preview, pre_render=False, post_render=True)

# pick the closest point
@scatter_plot.add_event_handler("click")
def on_pick(ev: pygfx.PointerEvent):
    global shader
    index = ev.pick_info["vertex_index"]
    shader_id = ids[index]
    scatter_plot.edge_colors = "black"  # reset all
    scatter_plot.edge_colors[index] = "white"
    shader = load_shadertoy(data["shaders"][index], PREVIEW_RES, device=fpl_device, canvas=canvas1)
    figure[1, 0].title = data["shaders"][index]["info"]["name"]
    print("loading shader:", shader)
    
    # we can actuall reset like this?
    figure[1,0].scene.clear()
    image = gfx.Image(
        gfx.Geometry(grid=texture1),
        gfx.ImageBasicMaterial(),
    )
    figure[1,0].scene.add(image)


    # print("picked on shader:", shader_id, url)


figure.show()

In [ ]:
# qt also closes gracefully
# figure.close()